# 12 From Prototype to Production

## 从 Notebook 原型走向工程系统，真正要补的不是部署脚本，而是治理结构

到这一章，这套项目在原型层面已经闭环了。前面已经建立了完整主线：为什么大模型会表现出任务性，为什么 control plane 决定行为边界，为什么 tool calling 是结构化动作出口，为什么 Agent 是任务运行时而 MCP 是能力层，为什么本地模型接入会暴露真实运行边界，为什么 Runtime 才是系统中枢，为什么案例和失败模式必须同时存在。

如果这是一套纯展示型 notebook，到这里其实已经够了。但如果要把这套东西进一步提升成“你不仅理解原理，还知道它怎么走向工程化”，就必须再回答最后一个问题：**从一个可解释、可运行的原型，到一个可扩展、可治理、可持续演进的系统，中间真正缺的是什么。**

很多人谈 production，第一反应是部署、容器、接口服务、监控面板。这些当然都重要，但它们通常不是 Agent 系统最先缺的东西。Agent 系统从原型走向工程化，最先暴露出来的往往不是“有没有上线方式”，而是“有没有稳定的治理结构”。

所以，这一章不会写成传统意义上的上线 checklist，而会把重点放在更关键的层面：能力如何扩展，权限如何收束，状态如何治理，提示词和资源如何版本化，trace 如何变成运维资产，系统如何在复杂度增长时仍然保持清晰。

## 先给结论

这一章最重要的判断可以压缩成一句话：

> Agent + MCP 原型走向生产，不是把 notebook 改成服务那么简单，而是把原本依赖作者个人控制的隐性秩序，升级成团队可维护、系统可审计、能力可治理的显性结构。

这句话里最关键的是“隐性秩序”这四个字。原型之所以能跑，很多时候依赖的是作者脑子里知道：

- 哪些能力该怎么用
- 哪些 prompt 该在什么任务下出现
- 哪些资源是可信输入
- 哪些错误可以先忽略

但一旦系统进入更真实的环境，这些“作者自己知道”的东西都必须外显成结构。否则，原型越成功，后续工程风险反而越大。

## 1. 原型为什么能跑，生产为什么会坏

很多原型在本地演示时表现很好，一到更真实的环境就开始不稳定，不是因为思路错了，而是因为原型默认了太多前提。

比如：

- 输入材料质量默认较高
- 用户请求默认比较克制
- 能力层规模默认很小
- Runtime 状态默认不会被并发或多任务污染
- prompt、tool、resource 的语义边界默认作者自己记得很清楚

这些前提在 notebook 阶段问题不大，因为任务规模小、上下文集中、操作者就是作者本人。但一旦进入更接近生产的环境，所有这些默认值都会失效。于是系统会呈现出一种很典型的现象：不是彻底不能用，而是越来越难以解释、难以维护、难以扩展。

这也是为什么 production 不是“把原型包起来发出去”，而是“把原型依赖的隐含规则显性化”。

## 2. 从原型到生产，最重要的架构变化是什么

最关键的变化，不是代码量变多，而是层次关系必须被收紧。

在原型里，很多层可以松散地粘在一起：

- Notebook 里直接写 prompt
- Runtime 里顺手定义能力元信息
- MCP Server 里顺手写一点任务逻辑
- 案例材料和正式 resource 混在一起

这种做法在原型阶段很高效，但一旦进入生产，层次就必须重新拉开。至少应该明确：

- 能力层只负责暴露能力，不负责任务编排
- Runtime 只负责任务推进，不负责私藏能力语义
- Prompt 模板、资源定义、工具契约都应成为独立治理对象
- 案例数据、测试数据和正式运行数据应分开管理

换句话说，原型到生产的第一步，不是加基础设施，而是去耦。

## 3. 多 Server 组织：能力规模一旦扩大，单一能力面会变脏

在 notebook 原型里，一个本地 MCP Server 足以承载整套演示，因为能力数量有限、边界清晰、任务集中。但生产化最大的变化之一，是能力规模会迅速扩大。

这时如果仍然坚持把所有 tools、resources、prompts 都塞进一个大而全的 server，问题会很快出现：

- 能力发现成本升高
- 描述风格开始不一致
- 权限边界难以收束
- 不同业务域之间相互污染

更合理的做法通常是按能力域拆分 server，例如：

- 文档与知识类 server
- 结构化分析类 server
- 业务操作类 server
- 团队私有模板与规范类 server

这并不是为了架构好看，而是为了让能力面在扩展时仍然保持可理解。对 Agent 来说，一个清晰分区的能力生态，远比一个堆满能力对象的巨大目录更容易稳定消费。

## 4. 权限治理：Agent 最危险的不是不会做事，而是做得太多

一旦系统进入更真实环境，权限问题会立刻变成核心问题。因为 Agent 与普通问答系统最大的差别之一，就是它不仅能说，还可能读、查、调、甚至触发真实动作。

这意味着能力层必须有明确的访问边界。例如：

- 哪些 resources 只是公开背景材料
- 哪些 resources 涉及敏感数据，必须限制任务场景或调用方身份
- 哪些 tools 只读，哪些 tools 可能有副作用
- 哪些 prompts 属于团队内部工作流，不应该对所有宿主暴露

如果这些边界不被明确，Agent 系统看起来越灵活，风险反而越高。因为系统最危险的状态，不是完全无能，而是不知道自己该被限制在哪。

## 5. Resource 版本化：上下文不是背景噪音，而是生产输入

很多人对代码版本化很敏感，但对资源版本化不够敏感。这在普通应用里可能问题不大，但在 Agent 系统里，resource 本身就是推理输入的一部分，因此它必须被当作正式输入治理。

如果一份资源内容变化了，可能带来什么后果？

- 模型后续判断路径改变
- 某些既有 prompt 模板不再匹配
- 某些工具输出解释前提失效
- 案例结果与历史 trace 不再可重现

这意味着 resource 不应该只是“某个文件还在那儿”，而应该有更明确的治理方式：

- 稳定 URI
- 版本标识
- 修改记录
- 对历史任务的可追溯关系

只有这样，系统的上下文层才不会随着资源变动悄悄漂移。

## 6. Prompt 版本化：提示词不是文案，而是控制逻辑

如果前面章节已经成立一个判断，即 prompt 是任务入口和控制逻辑的一部分，那么生产化时就必须顺势接受另一个判断：prompt 需要版本化。

原因非常直接。只要改动 prompt，就可能改变：

- 模型更倾向于直接回答还是优先走能力路径
- 输出结构更偏宽还是更偏收束
- 面对不确定信息时的保守程度
- 对相同资源和工具的使用倾向

这已经不是简单的措辞微调，而是任务控制逻辑变化。既然控制逻辑会影响系统行为，它就不该再被当作散落在代码里的随手字符串，而应成为可审查、可回滚、可比较的版本化对象。

这里最容易写空的一点，就是把 `tools / resources / prompts` 只列成三个名词，然后假装已经讲清治理了。真正有信息量的写法应该是：这三类东西一旦改动，系统会怎么变。

先说 `tools`。如果你改了 tool 的名字、输入契约或返回字段，影响的不是某个局部函数，而是模型对动作的选择倾向、Runtime 的解析逻辑，以及历史 trace 的可比性。比如原来 `score_candidate_fit` 返回的是维度分数，后来改成只返回总评，那么后面的综合结论为什么开始变虚，其实根源就在这里。

再说 `resources`。资源治理不是“文件有没有放好”，而是如果某份 PRD、岗位说明或架构文档更新了，系统后面的判断依据也会跟着换。没有稳定 URI、版本边界和可见范围，过一段时间你会连“为什么这个任务今天和上周跑出来不一样”都说不清。

最后是 `prompts`。这一层最容易被低估，因为很多人会把 prompt 改动看成措辞微调。但对 Agent 系统来说，prompt 变化经常等于控制逻辑变化。你把“信息不足时先读 resource”改弱一点，系统可能马上更爱直接回答；你把输出格式约束收紧一点，Runtime 的解析成功率又会变。它不是文案，而是行为开关。

所以把这三类对象拉出来单独治理，不是因为分类好看，而是因为它们分别控制着动作、依据和入口。系统一旦变复杂，不把这三样分开看，后面所有行为波动都会混成一团。

## 7. Runtime 状态治理：单任务原型与多任务系统不是一个难度级别

Notebook 原型通常默认一次只跑一个任务，状态结构也相对轻量。但只要系统进入更真实的服务环境，状态治理难度会立刻抬升。因为这时你面对的就不再是一个演示链条，而是多个任务、多个用户、多个执行轨迹并行存在。

这时至少会出现几类新问题：

- 哪些状态属于单任务工作记忆，任务结束后应被清理
- 哪些状态属于跨任务长期记忆，应被受控保留
- 如何避免不同任务之间的状态污染
- 失败中断的任务如何恢复或重放

这些问题在原型阶段几乎都可以先不处理，但生产化时如果没有明确答案，Runtime 会很快从“任务中枢”变成“隐含耦合的状态泥潭”。

## 8. Trace、Observability 与 Audit：原型的调试信息要升级成系统资产

在原型阶段，trace 的价值主要是为了作者自己看系统到底怎么跑。到了生产化阶段，trace 的角色会发生变化：它不再只是调试辅助，而会变成运行资产。

因为一旦系统真的持续运行，trace 至少承担三种不同职责：

- 调试：为什么某次任务失败或跑偏
- 评估：不同 prompt / tool / resource 版本下质量是否变化
- 审计：系统到底读了什么、调了什么、基于什么形成结论

特别是在 Agent 场景里，审计价值非常高。因为系统不是只输出一句答案，而是可能通过多步能力消费形成决策。没有可审计轨迹，很多看起来“合理”的行为其实无法事后解释。

## 9. 评估不应停留在 Notebook，而应进入持续运营能力

前一章已经建立了一个判断：Agent 评估不能只看 final answer，还要看路径、结构、状态和效率。如果系统要进一步走向工程化，这套评估框架就不能只停留在 notebook 说明里，而应该进入持续运营能力。

这意味着：

- 新的 prompt 版本上线前，应有对比验证
- 新增 tool 或 resource 后，应观察其是否改变路径合理性
- 模型版本变化后，应重新评估结构稳定性和工具倾向
- 某些典型任务应成为长期保留的 regression set

如果没有这套持续评估机制，生产化的 Agent 很容易表面上在迭代，实际上却在不断引入不可见的行为回归。

## 10. 成本、延迟与稳定性：生产化的真正权衡不是某个模型更强，而是系统如何平衡三者

原型阶段很容易只追求效果，觉得只要结果最好就行。但一旦进入工程环境，系统就必须开始面对更现实的三角关系：

- 模型越强，通常成本或延迟越高
- 约束越多、验证越严，稳定性更高，但吞吐和体验可能更差
- 能力层越丰富，任务成功率可能上升，但错误路径和治理成本也会增加

这就是为什么生产化不是简单地“把最强模型放进去”。更合理的做法往往是：根据任务复杂度、可接受延迟、风险等级来设计不同运行策略，而不是试图用单一配置覆盖所有场景。

## 11. 真正的生产化，最终会变成组织协作问题

很多技术系统到最后都会碰到一个类似事实：技术结构搭好之后，真正决定它能不能持续演进的，常常不再只是代码，而是协作机制。Agent + MCP 系统尤其如此。

原因在于，这类系统天然跨越多个对象：

- 模型选择与推理参数
- Prompt 模板与行为边界
- Tool / Resource / Prompt 能力定义
- Runtime 策略与错误恢复
- 评估数据与观测体系

这些对象往往不会只由一个人长期独占维护。因此，真正的生产化最终会落到一件事上：这些对象是否有清晰 ownership、变更流程和回归机制。没有这些，系统即使技术上很先进，也很难稳定长期演进。

## 12. 从原型走向生产时，最常见的错误是什么

从这套项目的视角看，几个最常见的错误大概是：

- 把 notebook 代码直接包成服务，却不重构层次
- 能力层越来越大，但没有 server 分区和权限治理
- prompt、resource、tool 一直改，但没有版本化和回归验证
- 把 trace 当临时日志，而不是系统资产
- 把模型效果波动一律归咎于模型，而不是反查 Runtime 和能力层设计

这些错误之所以常见，是因为它们都来自同一个误判：以为 production 只是运行环境升级，而不是系统治理升级。

## 13. 从作品集角度看，这一章到底在补什么

如果没有这一章，整套项目依然可以是一套很不错的技术说明和原型作品；但有了这一章，它就多了一个很重要的层次：你不仅知道怎么把系统搭出来，还知道它为什么很难被长期维持，以及应该从哪些结构入手去解决这种难。

也就是说，这一章补的不是新功能，而是工程判断：

- 你能识别原型成功背后的隐含秩序
- 你知道哪些秩序必须被显性化
- 你知道生产化的核心不只是部署，而是治理

对面试官和技术经理来说，这种判断通常比单纯多写几段代码更有价值。因为它说明你不是在做一次性演示，而是在理解系统生命周期。

## 14. 这套项目最终完成了什么

回头看整套 notebook，会发现它其实完成的是一条完整论证链：

- 从大模型的任务性外观出发
- 解释 control plane 如何塑形行为
- 解释 tool calling 如何成为动作出口
- 解释 Agent 为何是任务运行时
- 解释 MCP 为何是能力层而不是工具堆
- 解释本地模型、本地 server、Runtime、案例、评估如何拼成完整系统
- 最后再解释这套系统如果继续往前走，真正需要治理的是什么

这条链的意义在于，它让 `LLM -> Agent -> MCP -> Runtime -> Evaluation -> Production` 不再是几个流行词堆在一起，而是变成一个有因果、有边界、有工程落点的系统叙事。

## 15. 本章结论

这一章最值得保留的判断有这些：

- 原型走向生产，关键不是部署，而是把隐含秩序显性化。
- 多 server 组织、权限治理、resource/prompt 版本化、状态治理、trace 审计，都是能力系统继续扩展时绕不开的结构问题。
- 生产化不是“把 notebook 包起来”，而是重新梳理能力层、任务层和观测层的边界。
- 真正成熟的 Agent 系统，必须把评估和治理视为长期运行能力，而不是演示后补丁。
- 这套项目最终证明的，不只是会用 LLM 和 MCP，而是理解一套 Agent 系统从原理到工程生命周期的完整路径。

到这里，这套 notebook 才真正形成闭环。它不只是解释了 MCP 是什么，也不只是展示了 Agent 能做什么，而是把大模型、控制面、能力层、运行时、案例、评估和工程化全部压进了一条统一的系统叙事里。